# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Usaf007/flyrankai-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### 1. Unit of Analysis & Time Window

* **Unit of Analysis (Grain):** One row represents one unique pseudonymized content item (`content_hash_id`) for a specific client (`client_hash_id`) on a single report day (`report_date`).
* **Time Window:** Mid-panel month `month=2026-03` (March 1, 2026 to March 31, 2026).
* **Justification:** Developing feature logic on a mid-panel month prevents peeking into future outcome windows (such as the final month `2026-06`), keeping our evaluation leak-free.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
import pandas as pd
from google.colab import userdata

# Connecting and Authenticating
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}');")

rel = "hf://datasets/FlyRank/internship-warehouse"
fact_table = f"{rel}/fact_content_daily_performance/month=2026-03/*.parquet"

# Verifying time window boundaries
window_check = con.execute(f"""
    SELECT
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date,
        COUNT(DISTINCT report_date) AS total_days
    FROM read_parquet('{fact_table}')
""").df()

print("--- TIME WINDOW VERIFICATION ---")
print(window_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- TIME WINDOW VERIFICATION ---
  start_date   end_date  total_days
0 2026-03-01 2026-03-31          31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### 2. Field Classification Matrix

| Bucket | Field Name(s) | Role & Justification |
| :--- | :--- | :--- |
| **Features** | `gsc_impressions`, `ga4_sessions`, `gsc_clicks`, `content_age_days`, `word_count` | Observable search/analytics measurements knowable prior to the decision point. |
| **Label / Proxy** | `target_is_declining` | Binary target indicating lost rankings (`gsc_avg_position > 10`), serving as our decay proxy in the raw warehouse. |
| **Context** | `client_hash_id`, `content_hash_id`, `report_date` | Used strictly for joins, grouping, and segmenting during evaluation. |
| **Excluded** | `health_score`, `priority_score`, `action_type`, product flags | **Why:** These are product decisions computed by hand-tuned SQL. Including them creates circular results where the model merely memorizes app rules instead of learning patterns from raw signals. |

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

dim_table = f"{rel}/dim_content.parquet"

# Calculating content_age_days dynamically and use average position as the decay proxy
query = f"""
    SELECT
        f.gsc_impressions,
        f.ga4_sessions,
        f.gsc_clicks,
        date_diff('day', d.content_created_date, f.report_date) AS content_age_days,
        d.word_count,
        CAST(CASE WHEN f.gsc_avg_position > 10 THEN 1 ELSE 0 END AS INTEGER) AS target_is_declining
    FROM read_parquet('{fact_table}') f
    JOIN read_parquet('{dim_table}') d ON f.content_hash_id = d.content_hash_id
    WHERE f.ga4_data_available IS TRUE
    LIMIT 5000
"""
df_features = con.execute(query).df().dropna()

# --- THE LEAKAGE TRAP ---
# Deliberately creating a column that secretly contains the answer key
df_features['leaked_column'] = df_features['target_is_declining']

X_leak = df_features[['gsc_impressions', 'ga4_sessions', 'gsc_clicks', 'content_age_days', 'word_count', 'leaked_column']]
y = df_features['target_is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X_leak, y, test_size=0.2, random_state=42)
clf = RandomForestClassifier(random_state=42).fit(X_tr, y_tr)
print(f"Score WITH Leaked Feature (The Trap): {accuracy_score(y_te, clf.predict(X_te))*100:.2f}%")

# Honest Score (Leak Removed)
X_honest = df_features[['gsc_impressions', 'ga4_sessions', 'gsc_clicks', 'content_age_days', 'word_count']]
X_tr_h, X_te_h, y_tr_h, y_te_h = train_test_split(X_honest, y, test_size=0.2, random_state=42)
clf_h = RandomForestClassifier(random_state=42).fit(X_tr_h, y_tr_h)
print(f"Score WITHOUT Leaked Feature (Honest Model): {accuracy_score(y_te_h, clf_h.predict(X_te_h))*100:.2f}%")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Score WITH Leaked Feature (The Trap): 100.00%
Score WITHOUT Leaked Feature (Honest Model): 79.45%


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### 3. Verification Queries

We verify three core claims using DuckDB queries on our mid-panel slice (`month=2026-03`):
1. **Grain Uniqueness:** Proving composite keys (`report_date` + `client_hash_id` + `content_hash_id`) contain zero duplicates.
2. **Row Count & Volume:** Measuring the total fact records available in March 2026.
3. **Data Availability Filter:** Checking row survival when filtering explicitly with `ga4_data_available IS TRUE`.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("--- QUERY 1: GRAIN UNIQUENESS CHECK ---")
grain_check = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT report_date || client_hash_id || content_hash_id) AS unique_composite_keys
    FROM read_parquet('{fact_table}')
    LIMIT 100000
""").df()
print(grain_check)

print("\n--- QUERY 2: MARCH 2026 ROW COUNT & SPAN ---")
counts_check = con.execute(f"""
    SELECT
        COUNT(*) AS march_total_rows,
        COUNT(DISTINCT client_hash_id) AS active_clients,
        COUNT(DISTINCT content_hash_id) AS active_content_items
    FROM read_parquet('{fact_table}')
""").df()
print(counts_check)

print("\n--- QUERY 3: AVAILABILITY FILTER (IS TRUE) ---")
avail_check = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        ROUND(AVG(CASE WHEN ga4_data_available IS TRUE THEN 1.0 ELSE 0.0 END) * 100, 2) AS availability_pct
    FROM read_parquet('{fact_table}')
""").df()
print(avail_check)

--- QUERY 1: GRAIN UNIQUENESS CHECK ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_composite_keys
0     9841378                9841378

--- QUERY 2: MARCH 2026 ROW COUNT & SPAN ---
   march_total_rows  active_clients  active_content_items
0           9841378              55                331437

--- QUERY 3: AVAILABILITY FILTER (IS TRUE) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  ga4_available_rows  availability_pct
0     9841378            413966.0              4.21


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### 4. Data Limits & Constraints

1. **Unbalanced History:** History coverage varies by client depending on when tracking started (`gsc_data_start` vs `ga4_data_start`).
2. **GSC-Only Rows:** Earlier dates contain search data only, where `ga4_data_available` is `FALSE`. Treating missing GA4 metrics as zero traffic without filtering leads to false decay signals.
3. **No Causality:** This observational dataset records natural performance movements. It cannot guarantee that refreshing a flagged page will directly cause traffic recovery.
4. **Single-Month Seasonality Loss:** Evaluating a single month (`month=2026-03`) loses annual seasonal context (e.g., holiday or industry-specific search shifts).

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

rel = "hf://datasets/FlyRank/internship-warehouse"
dim_clients_table = f"{rel}/dim_clients.parquet"
fact_table = f"{rel}/fact_content_daily_performance/month=2026-03/*.parquet"

limit_check = con.execute(f"""
    SELECT
        c.client_hash_id,
        c.gsc_data_start,
        c.ga4_data_start,
        COUNT(f.report_date) AS march_fact_rows
    FROM read_parquet('{dim_clients_table}') c
    LEFT JOIN read_parquet('{fact_table}') f ON c.client_hash_id = f.client_hash_id
    GROUP BY c.client_hash_id, c.gsc_data_start, c.ga4_data_start
    LIMIT 10
""").df()

print("--- DATA LIMITATION DEMONSTRATION (Unbalanced Client History) ---")
print(limit_check)

--- DATA LIMITATION DEMONSTRATION (Unbalanced Client History) ---
            client_hash_id gsc_data_start ga4_data_start  march_fact_rows
0  client_65de48885f4ef01b     2025-06-21     2026-02-19           426307
1  client_c182d11e4862a37d     2025-06-21     2026-02-20            57784
2  client_a2eeb8899886adde     2025-07-06     2026-02-19            36538
3  client_764ae36a94e30a25     2025-11-07     2026-02-19             3751
4  client_59256b0571e0c970     2025-09-24     2026-02-19             3441
5  client_795153d5b7850ccf     2025-09-24            NaT            78709
6  client_fef1a8f436438636     2025-03-11     2026-03-06           335379
7  client_3197e6291363b4db     2025-06-29     2025-11-09           316610
8  client_ba65e80a1116ae41     2025-10-13     2025-11-09           410409
9  client_2094c6eb080311d5     2025-12-17     2025-12-18           173700


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.